# Trading Strategies Module

**Quantitative Research Portfolio - Module 2**

**Author**: Kevin J. Metzler

---

## Purpose

This notebook demonstrates the strategy implementations in `02_trading_strategies`. It covers:
- Mean reversion with Ornstein-Uhlenbeck modeling
- Cross-sectional momentum
- Statistical arbitrage using cointegration

The workflow uses synthetic price data to illustrate signal generation, position sizing, and backtesting.

## Mathematical Outline

**Mean Reversion (OU Process)**

$$dX_t = \theta(\mu - X_t)dt + \sigma dW_t$$

**Momentum Signal**

$$r_{i,t+1} = \alpha + \beta r_{i,t-k:t} + \epsilon_{i,t+1}$$

**Statistical Arbitrage (Cointegration)**

$$P_{1,t} - \beta P_{2,t} \sim I(0)$$

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath("."))

from trading_strategies import (
    StrategyConfig,
    MeanReversionStrategy,
    MomentumStrategy,
    StatisticalArbitrageStrategy,
)

sns.set_style("whitegrid")
%matplotlib inline

## Synthetic Market Data

In [ ]:
def simulate_prices(n_assets=6, n_days=400, seed=42):
    rng = np.random.default_rng(seed)
    mu = 0.0004
    vol = 0.02
    corr = 0.3

    cov = (1 - corr) * np.eye(n_assets) + corr * np.ones((n_assets, n_assets))
    chol = np.linalg.cholesky(cov)
    shocks = rng.standard_normal((n_days, n_assets)) @ chol.T
    returns = mu + vol * shocks

    dates = pd.date_range("2018-01-01", periods=n_days, freq="B")
    prices = 100 * np.exp(np.cumsum(returns, axis=0))
    columns = [f"Asset_{i+1}" for i in range(n_assets)]
    return pd.DataFrame(prices, index=dates, columns=columns)


prices = simulate_prices()
returns = prices.pct_change().dropna()
benchmark = returns.mean(axis=1)

In [ ]:
prices.plot(figsize=(10, 4), title="Synthetic Price Series")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

## Strategy Configuration

In [ ]:
configs = {
    "mean_reversion": StrategyConfig(
        name="Mean Reversion",
        lookback_period=120,
        holding_period=5,
        transaction_cost=0.001,
        max_position_size=0.2,
    ),
    "momentum": StrategyConfig(
        name="Momentum",
        lookback_period=120,
        holding_period=20,
        transaction_cost=0.001,
        max_position_size=0.2,
    ),
    "stat_arb": StrategyConfig(
        name="Statistical Arbitrage",
        lookback_period=120,
        holding_period=5,
        transaction_cost=0.0005,
        max_position_size=0.2,
    ),
}

strategies = {
    "Mean Reversion": MeanReversionStrategy(configs["mean_reversion"]),
    "Momentum": MomentumStrategy(configs["momentum"], use_ml_enhancement=False),
    "Statistical Arbitrage": StatisticalArbitrageStrategy(
        configs["stat_arb"], cointegration_lookback=120
    ),
}

## Backtesting

In [ ]:
results = {}
for name, strategy in strategies.items():
    results[name] = strategy.backtest(prices, benchmark)

metrics = [
    "annualized_return",
    "volatility",
    "sharpe_ratio",
    "max_drawdown",
    "calmar_ratio",
]

performance_table = pd.DataFrame(
    {name: {m: res["performance"][m] for m in metrics} for name, res in results.items()}
).T
performance_table

In [ ]:
def cumulative_returns(series):
    return (1 + series).cumprod()

plt.figure(figsize=(10, 5))
for name, res in results.items():
    cum = cumulative_returns(res["returns"])
    plt.plot(cum.index, cum.values, label=name)

bench_cum = cumulative_returns(benchmark.loc[cum.index])
plt.plot(bench_cum.index, bench_cum.values, label="Benchmark", linestyle="--")

plt.title("Cumulative Returns")
plt.legend()
plt.show()

In [ ]:
def drawdown(series):
    cumulative = cumulative_returns(series)
    peak = cumulative.cummax()
    return (cumulative - peak) / peak

plt.figure(figsize=(10, 5))
for name, res in results.items():
    dd = drawdown(res["returns"])
    plt.plot(dd.index, dd.values, label=name)

plt.title("Drawdowns")
plt.legend()
plt.show()

## Sample Positions

In [ ]:
momentum_positions = results["Momentum"]["positions"]
momentum_positions.tail()

## Summary

This notebook demonstrates the main strategy classes, shows how to run backtests, and compares outcomes across strategies. The same workflow can be applied to real market data by replacing the synthetic price series.